# Tensors & Autograd

> 📘 **Python Mastery** · Module 14 — Deep Learning · Lesson 2/7

Tensors are NumPy arrays with GPU superpowers and a photographic memory of how every number was computed — that memory is *autograd*, the engine that trains every neural network you will ever build.

## 🎯 Learning Objectives

- **Classify** scalars, vectors, matrices, and 3D/4D tensors and interpret shapes like `(batch, channels, height, width)`.
- **Create** tensors from nested lists and factory functions, translating freely between NumPy and PyTorch syntax.
- **Reshape, transpose/permute, and broadcast** tensors using rules identical in both libraries.
- **Choose** correct dtypes (`float32` weights, `int64` labels) and explain why deep learning defaults to `float32`.
- **Explain** what autograd does: `requires_grad`, `backward()`, `.grad`, and `torch.no_grad()`.
- **Verify** a gradient by hand with a numerical check in NumPy — the same value autograd returns.

## 1. What Is a Tensor?

A tensor is just an N-dimensional array — exactly what NumPy gives you — plus two extras deep learning needs: fast GPU execution and automatic gradients. The word only sounds scary; you have used tensors since Module 10.

| Rank | Name | Shape example | ML meaning |
|---|---|---|---|
| 0 | Scalar | `()` | A single loss value |
| 1 | Vector | `(5,)` | One sample with 5 features |
| 2 | Matrix | `(32, 10)` | Batch of 32 samples × 10 features |
| 3 | 3D tensor | `(3, 224, 224)` | One RGB image (channels, height, width) |
| 4 | 4D tensor | `(8, 3, 224, 224)` | **Batch of 8 images**: B, C, H, W |

Reading shapes is a core engineering skill. `(8, 3, 224, 224)` is pronounced "batch of eight, three colour channels, 224 pixels tall, 224 wide".

**Syntax:** (NumPy today, PyTorch mirror below)
```python
import numpy as np
t = np.array([[1., 2., 3.],
              [4., 5., 6.]])   # rank-2 tensor
t.ndim, t.shape, t.dtype        # dimension count, size per axis, cell type
```

In [ ]:
import numpy as np

scalar   = np.array(3.14)                       # rank 0
vector   = np.array([70., 180., 25.])           # rank 1: one patient's features
matrix   = np.array([[70., 180., 25.],          # rank 2: three patients
                     [65., 170., 31.],
                     [90., 175., 28.]])
image    = np.zeros((3, 28, 28))                # rank 3: one grayscale-RGB image
batch    = np.zeros((8, 3, 28, 28))             # rank 4: batch of images

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix),
                ("image (C,H,W)", image), ("batch (B,C,H,W)", batch)]:
    print(f"{name:16s} ndim={t.ndim}  shape={t.shape}  dtype={t.dtype}")

# The letters B C H W are how CNNs (Lesson 5) describe image batches:
B, C, H, W = batch.shape
print(f"\nB={B} images, C={C} channels, H={H} rows, W={W} cols "
      f"-> {batch.nbytes / 1024:.1f} KB as float64")

**PyTorch version** (requires `pip install torch`)

Same object, new type: `torch.Tensor`.

```python
import torch

t = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])     # from a nested list
print(t.ndim, t.shape, t.dtype)      # 2 torch.Size([2, 3]) torch.float32
img  = torch.zeros(3, 224, 224)      # factory functions too
batch = torch.zeros(8, 3, 224, 224)
```

`torch.tensor` copies data; `torch.from_numpy(arr)` wraps an existing NumPy array (shares memory, zero copy).

## 2. Creating Tensors: a Two-Way Dictionary

Everything you know from NumPy has a PyTorch twin with the same name and behaviour:

| You want | NumPy | PyTorch |
|---|---|---|
| From nested lists | `np.array([[1, 2]])` | `torch.tensor([[1, 2]])` |
| All zeros / ones | `np.zeros((3, 4))`, `np.ones((3, 4))` | `torch.zeros(3, 4)`, `torch.ones(3, 4)` |
| Range / evenly spaced | `np.arange(0, 10, 2)`, `np.linspace(0, 1, 5)` | `torch.arange(0, 10, 2)`, `torch.linspace(0, 1, 5)` |
| Random normal / uniform | `rng.normal(size=(2, 3))` | `torch.randn(2, 3)`, `torch.rand(2, 3)` |
| Identity matrix | `np.eye(3)` | `torch.eye(3)` |
| Same shape as another | `np.zeros_like(a)` | `torch.zeros_like(a)` |

One quirk to notice: PyTorch factory functions take each dimension as a separate argument (`torch.zeros(3, 4)`), while NumPy takes a shape tuple.

**Syntax:**
```python
weights = rng.normal(0, 0.01, size=(n_features, n_neurons))   # NumPy
weights = torch.randn(n_features, n_neurons) * 0.01           # PyTorch equivalent
```

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

a = np.array([[1., 2., 3.],
              [4., 5., 6.]])
print("from lists:\n", a)

print("zeros:", np.zeros((2, 3)).shape, "| ones:", np.ones(4))
print("arange:", np.arange(0, 10, 2))
print("linspace:", np.linspace(0, 1, 5).round(2))

w_init = rng.normal(0, 0.01, size=(3, 2))       # typical weight init
print("random weights (seeded):\n", w_init.round(4))

b = np.zeros(a.shape[1])
print("zeros_like(a.T):", np.zeros_like(a.T).shape, "<- matches layer output width")

**PyTorch version** (requires `pip install torch`)
```python
import torch
torch.manual_seed(42)                 # seeding, NumPy-style reproducibility

a  = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
z  = torch.zeros(2, 3); o = torch.ones(4)
ar = torch.arange(0, 10, 2); ls = torch.linspace(0, 1, 5)
w  = torch.randn(3, 2) * 0.01         # weight init
b  = torch.zeros(2)
zb = torch.zeros_like(a.T)            # same-shape helper
```

## 3. Shape Surgery: reshape / view / -1 / transpose / permute

Networks constantly rearrange data without touching the underlying bytes: flatten an image batch into rows, swap channel order, turn `(B, C, H, W)` into `(B, H*W, C)` for attention layers. Reshaping only changes how the same memory buffer is *interpreted*, which is why it is essentially free.

- `reshape` / `view` — regroup elements into a new shape (`view` requires compatible memory layout; `reshape` copies if needed).
- `-1` — "you figure this axis out": `arr.reshape(32, -1)`.
- `transpose(axes)` (NumPy) / `permute(dims)` (PyTorch) — reorder axes.
- `.T` — shortcut that reverses all axes.

**Syntax:**
```python
flat = arr.reshape(arr.shape[0], -1)      # (B, C, H, W) -> (B, C*H*W)
nhwc = img.transpose(1, 2, 0)             # CHW -> HWC for matplotlib
x2   = x.permute(0, 2, 3, 1)              # same idea in torch
```

In [ ]:
import numpy as np

grades = np.arange(12, dtype=float).reshape(3, 4)   # 3 students x 4 subjects
print("grades:\n", grades)

print("reshape(-1) flattens ->", grades.reshape(-1))
print("reshape(4, 3) transposes roles:\n", grades.reshape(4, 3))

# -1 lets ONE axis be inferred:
print("reshape(6, -1).shape =", grades.reshape(6, -1).shape)

# transpose: swap axes (rows <-> columns here)
print("grades.T shape:", grades.T.shape, "(now 4 subjects x 3 students)")

# permute on an image batch: (B, C, H, W) -> (B, H, W, C)
imgs = np.random.default_rng(0).random((2, 3, 4, 5))
as_hwc_batch = imgs.transpose(0, 2, 3, 1)           # numpy takes AXIS INDICES
print("NHWC batch:", as_hwc_batch.shape, "-> matplotlib-ready per-image shape",
      as_hwc_batch[0].shape)

**PyTorch version** (requires `pip install torch`)
```python
import torch
g = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print(g.reshape(-1).shape)          # torch.Size([12])   like reshape(-1)
print(g.view(4, 3).shape)           # view: free reshape when memory allows
print(g.t().shape)                  # .T shortcut
imgs = torch.rand(2, 3, 4, 5)
print(imgs.permute(0, 2, 3, 1).shape)  # torch.Size([2, 4, 5, 3]); note DIMS not axes
```

Gotcha: NumPy's `transpose(0, 2, 3, 1)` and torch's `permute(0, 2, 3, 1)` match, but torch also has `transpose(dim0, dim1)` for swapping just two axes.

> 🔍 **Under the Hood:** reshaping never moves data — it writes a new *stride table*. `arr.reshape(4, 3)` still points at the same 12 floats; only "how far to jump for the next row" changed. `transpose` likewise just rewrites strides, which is why it is instant but makes later operations slower (memory is no longer read contiguously). `view` refuses when the stride trick cannot work; `reshape` then silently falls back to copying.

## 4. Broadcasting: One Rulebook for Both Libraries

Broadcasting stretches arrays of different shapes into a common shape *without copying data*. The rule (identical in NumPy and PyTorch):

1. Compare shapes **right-to-left**, padding the shorter with leading 1s.
2. Each pair of dims must be **equal**, or one of them must be **1** (that one gets stretched).
3. Any missing leading dims behave as if they were 1.

This is why `X @ W + b` works even though `X` is `(200, 4)` and `b` is `(4,)`: right-aligned, `4 == 4`, so the bias row is stretched over all 200 samples.

**Syntax:**
```python
centered = X - X.mean(axis=0)          # (N, F) minus (F,)  -> (N, F)
scores   = col_vec + row_vec           # (N, 1) + (1, M)    -> (N, M)
```

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

# Real scenario: standardise exam scores per subject across students.
scores = rng.integers(40, 100, size=(5, 4)).astype(float)   # 5 students, 4 subjects
mean_per_subject = scores.mean(axis=0)                      # shape (4,)
std_per_subject  = scores.std(axis=0)                       # shape (4,)
standardised = (scores - mean_per_subject) / std_per_subject
print("raw means :", mean_per_subject.round(1))
print("(5,) broadcasts against (5, 4) column-by-column ->")
print("standardised column means now:", standardised.mean(axis=0).round(6))

# Explicit dim trick: add a NEW axis to broadcast down rows instead.
row_bias = np.array([[0.5], [1.0], [1.5], [2.0], [2.5]])    # (5, 1)
curved = scores + row_bias                                  # (5,1)+(5,4)->(5,4)
print("per-student curve applied, shape kept:", curved.shape)

# The classic outer-product style broadcast: (3,1) + (1,4) -> (3,4)
grid = np.arange(3).reshape(3, 1) + np.arange(4).reshape(1, 4)
print("outer sum:\n", grid)

**PyTorch version** (requires `pip install torch`) — identical rules, identical result:
```python
import torch
scores = torch.randint(40, 100, (5, 4)).float()
standardised = (scores - scores.mean(0)) / scores.std(0)     # (5,4)-(4,)
curved = scores + torch.tensor([0.5, 1.0]).unsqueeze(1)      # unsqueeze == None-axis
outer  = torch.arange(3).unsqueeze(1) + torch.arange(4)      # (3,1)+(4,) -> (3,4)
```

`tensor.unsqueeze(1)` / `tensor[:, None]` insert a length-1 axis — your main tool for *aiming* a broadcast.

## 5. dtypes & Devices: Precision and Location

Deep learning defaults to **float32** ("single precision"), not float64, because:

- GPUs pack twice as many float32 lanes per chip → roughly **2x throughput**;
- half the memory traffic and VRAM (models barely fit as it is);
- training tolerates ~7 decimal digits fine — gradients are noisy anyway.

Labels/class indices use **int64** (`torch.long`). Mixing dtypes mid-computation raises errors in torch where NumPy would quietly upcast — arguably a feature.

The **device** is where the tensor lives: `"cpu"` or `"cuda:N"` (an NVIDIA GPU). Tensors must sit on the same device to interact; moving data CPU↔GPU is the slow part of GPU training, so you move it once, up front.

**Syntax:**
```python
w = w.astype(np.float32)              # NumPy cast
w = w.to(torch.float32)               # torch cast
w = w.to("cuda")                     # torch device move (GPU required)
```

In [ ]:
import numpy as np

w64 = np.random.default_rng(0).normal(size=(500, 500))     # float64 default
w32 = w64.astype(np.float32)
w16 = w64.astype(np.float16)

for name, m in [("float64", w64), ("float32", w32), ("float16", w16)]:
    print(f"{name}: itemsize={m.itemsize} byte(s), "
          f"a (10000,10000) matrix = {m.itemsize * 1e8 / 1e6:,.0f} MB")

labels  = np.array([0, 1, 2, 1], dtype=np.int64)   # class indices stay int64
logits  = np.array([0.1, 2.3, -1.0, 0.7], dtype=np.float32)
print("labels:", labels.dtype, "| logits:", logits.dtype)

# float32 keeps ~7 significant digits -- plenty for gradients:
big = np.array([123456789.0], dtype=np.float32)
small = np.array([1e-8], dtype=np.float32)
print("float32 loses tiny additions:", big + small == big, "(float64 would keep them)")

**PyTorch version** (requires `pip install torch`)
```python
import torch
w = torch.randn(500, 500)          # already float32!
print(w.dtype, w.device)           # torch.float32 cpu
w_gpu = w.to("cuda")               # move once; everything after runs on GPU
model = model.to("cuda")           # models move the same way
labels = torch.tensor([0, 1, 2], dtype=torch.long)
mixed = w.half()                   # float16 for mixed-precision training
```

No GPU in this environment? Every lesson here runs identically on `device="cpu"` — just slower for big models.

## 6. Indexing & Slicing: Parity Across Libraries

If you can slice a NumPy array you can slice a tensor — same syntax, same views-vs-copies semantics:

| Task | Code (works in both, modulo `tensor(...)` vs `array(...)`) |
|---|---|
| First 3 rows | `x[:3]` |
| Last column | `x[:, -1]` |
| Every other row | `x[::2]` |
| Boolean mask | `x[x > 50]` |
| Fancy index rows | `x[[0, 2, 4]]` |
| Reverse order | `x[::-1]` |

Slices return **views** (mutations show through) while boolean/fancy indexing returns **copies** — true in NumPy and PyTorch alike.

**Syntax:**
```python
adults   = ages[ages >= 18]          # filter
subset   = X[:100, :5]               # window
picked   = names[[0, 2]]             # gather
```

In [ ]:
import numpy as np

patients = np.array([[72., 120., 98.],     # heart rate, systolic BP, SpO2
                     [95., 150., 91.],
                     [68., 118., 99.],
                     [110., 160., 88.]])

print("first two patients:\n", patients[:2])
print("BP column:", patients[:, 1])

flagged = patients[patients[:, 0] > 90]         # boolean mask: high heart rate
print("high-risk rows:\n", flagged)

worst = patients[np.argmax(patients[:, 2])]     # fancy index: lowest SpO2 row
print("lowest oxygen saturation row:", worst)
print("reversed:", patients[::-1, 0])

**PyTorch version** (requires `pip install torch`) — literally the same expressions:
```python
import torch
p = torch.tensor([[72., 120., 98.], [95., 150., 91.], [68., 118., 99.], [110., 160., 88.]])
print(p[:2], p[:, 1], sep="\n")
print(p[p[:, 0] > 90])
print(p[torch.argmin(p[:, 2])])   # row with lowest SpO2
```
One difference: converting a torch slice back to NumPy uses `p.numpy()` (CPU tensors only; GPU first `.cpu()`).

## 7. Autograd: the Framework Does Lesson 1's Chain Rule For You

Recall the backward pass we derived by hand. Autograd generalises it: every operation on a tensor marked `requires_grad=True` is recorded into a computational graph. Call `.backward()` on any scalar output and the framework walks that graph in reverse, depositing exact derivatives into `tensor.grad`. No hand-derived formulas, ever again.

The four verbs you will use daily:

| Verb | Meaning |
|---|---|
| `x.requires_grad = True` | Start recording operations involving `x` (parameters get this automatically inside `nn.Module`) |
| `loss.backward()` | Populate `.grad` everywhere upstream via reverse-mode differentiation |
| `optimizer.zero_grad()` | Clear stale `.grad` values — they **accumulate** by design |
| `with torch.no_grad():` | Stop recording — used at inference/validation to save memory and time |

**PyTorch version** (requires `pip install torch`)
```python
import torch

X = torch.randn(30, 4)
y = torch.randn(30, 1)
w = torch.randn(4, 1, requires_grad=True)      # <- track this leaf

loss = ((X @ w - y) ** 2).mean()               # forward pass builds the graph
loss.backward()                                 # reverse pass fills w.grad
print(w.grad)                                   # EXACT d(loss)/dw, no formulas written

w.grad.zero_()                                  # clear before the next step (it accumulates!)

with torch.no_grad():                           # inference mode
    preds = X @ w                               # graph NOT recorded -> fast, light
```

> 🔍 **Under the Hood:** autograd builds a directed acyclic graph of `Function` nodes as your forward line executes (define-by-run). Each node stores just enough to compute local derivatives — e.g. the mask `X > 0` saved by ReLU. `.grad` **accumulates** across calls because gradients of shared sub-graphs must be summed; that design choice is why forgetting `zero_grad()` silently doubles your updates. `no_grad()` simply skips node creation, cutting memory roughly in half during inference.

## 8. Proof by Hand: Numerical Gradients in NumPy

You do not need torch installed to see exactly what autograd computes. For $f(w) = \mathrm{mean}\big((wX - y)^2\big)$, nudge each weight ±$\epsilon$ and measure the slope:

$$\frac{\partial f}{\partial w_i} \approx \frac{f(w+\epsilon e_i) - f(w-\epsilon e_i)}{2\epsilon}$$

This **central difference** estimate should agree with the analytic gradient $\frac{2}{N} X^\top(Xw-y)$ to ~8 decimal places. When your hand-derived gradients match this, you have reproduced autograd itself — a debugging technique (*gradient checking*) used on real research code.

**Syntax:**
```python
num_grad[i] = (f(w + eps*e_i) - f(w - eps*e_i)) / (2*eps)   # slope probe
ana_grad    = 2/N * X.T @ (X @ w - y)                        # calculus answer
```

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
X = rng.normal(size=(30, 4))                # 30 samples, 4 features
y = rng.normal(size=(30, 1))
w = rng.normal(scale=0.1, size=(4, 1))

def f(w):                                   # loss whose gradient autograd would find
    return np.mean((X @ w - y) ** 2)

def analytic_grad(w):
    return 2 / len(y) * X.T @ (X @ w - y)   # chain rule, done once by us

eps = 1e-6
num_grad = np.zeros_like(w)
it = np.nditer(w, flags=["multi_index"])
while not it.finished:                      # probe EVERY entry independently
    i = it.multi_index
    wp, wm = w.copy(), w.copy()
    wp[i] += eps; wm[i] -= eps
    num_grad[i] = (f(wp) - f(wm)) / (2 * eps)
    it.iternext()

ana = analytic_grad(w)
print("numerical grad :", num_grad.ravel().round(8))
print("analytic grad  :", ana.ravel().round(8))
print("max abs difference:", np.abs(num_grad - ana).max())
print("\n<- autograd returns exactly these numbers, for ANY computation graph.")

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Reading `.grad` before calling `.backward()` | It is `None` — nothing computed yet | Forward → `loss.backward()` → then inspect `.grad` |
| Forgetting `optimizer.zero_grad()` | Gradients accumulate and updates grow wildly each step | Zero at the top of every training loop iteration |
| Mixing float64 and float32 in torch | Hard error instead of silent upcast | Cast inputs: `X.to(torch.float32)`; keep one dtype everywhere |
| Assuming `view`/`reshape` always copy (or never do) | Surprise mutations through views | Views share memory; use `.clone()` when independence matters |
| Training under `no_grad` or inferring without it | First: no learning. Second: wasted memory/time | Wrap *validation/inference* in `with torch.no_grad():`; train outside it |
| Device mismatch (`cuda` tensor + `cpu` tensor) | Runtime error about expected devices | Move model AND batch: `model.to(device)`, `X.to(device)` |

## 💡 Best Practices & Pro Tips

- **Print `.shape` at every unfamiliar step.** 90% of deep-learning bugs are shape bugs, caught instantly by eyeballing dimensions.
- **Standardise on float32 for models, int64 for labels** — the convention every tutorial, checkpoint, and GPU assumes.
- **Aim broadcasts explicitly** with `[:, None]` / `unsqueeze(1)` rather than relying on accidental alignment.
- **Detach before plotting:** convert to plain numbers with `arr.detach().cpu().numpy()` (or just `.numpy()`) so matplotlib never drags the autograd graph along.
- **Gradient-check custom layers once:** compare analytic vs numerical gradients (Section 8) whenever you write novel maths — it catches sign errors that silently cripple learning.
- **AI-engineering relevance:** shape/dtype discipline is precisely what breaks in production pipelines — mixed precision, ONNX exports, and batch-size changes all surface as tensor bugs, not algorithm bugs.

## 📌 Summary

| Concept | What it does | Example |
|---|---|---|
| Tensor | N-d array (+GPU, +autograd) | `torch.tensor([[1., 2.]])` vs `np.array([[1., 2.]])` |
| Shape surgery | Reinterpret memory cheaply | `x.reshape(B, -1)`, `x.permute(0, 2, 3, 1)` |
| Broadcasting | Stretch shapes without copies | `(X - X.mean(0)) / X.std(0)` |
| dtype choice | Speed/memory vs precision | `float32` params, `int64` labels |
| `requires_grad` | Record ops for differentiation | `w = torch.randn(4, 1, requires_grad=True)` |
| `loss.backward()` | Fill `.grad` via chain rule | then `optimizer.step()` |
| Numerical gradient | Ground-truth check | `(f(w+h) - f(w-h)) / (2*h)` |

Key takeaways:
- Tensors ARE NumPy arrays semantically — every slicing/broadcasting skill transfers 1:1.
- `float32` is the DL default for speed and memory; labels ride along as `int64`.
- Autograd = your Lesson-1 chain rule automated over a recorded graph; `.grad` accumulates until cleared.
- Central-difference numerical gradients let you verify any derivative claim without a framework.

## 🔗 Next Lesson

**03_First_Neural_Network** — combine tensors, activations, losses, and gradients to train a real classifier on a non-linear dataset, then watch its decision boundary emerge.